[Reference](https://medium.com/@shubhamnagar234/retrievers-in-langchain-how-to-retrieve-relevant-documents-part-13-ce1412a0887e$0)

# Wikipedia Retriever
```
User Query
    ↓
Wikipedia Retriever
    ↓
Wikipedia API
    ↓
Relevant Wikipedia Articles
    ↓
Content Extraction
    ↓
LangChain Document Objects
```

In [1]:
from langchain_community.retrievers import WikipediaRetriever

# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang="en")

# Define your query
query = (
    "the geopolitical history of india and pakistan from the perspective of a chinese"
)

# Get relevant Wikipedia documents
docs = retriever.invoke(query)

# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")

# Vector Store Retriever
```
Documents
    ↓
Embedding Model
    ↓
Document Vectors
    ↓
Vector Store
    │
    │
User Query
    ↓
Embedding Model
    ↓
Query Vector
    ↓
Similarity Search
    ↓
Top-k Relevant Documents
```

In [2]:
# If using OpenAIEmbeddings
# from langchain_openai import OpenAIEmbeddings

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv

load_dotenv()

# Step 1: Your source documents
documents = [
    Document(
        page_content="LangChain helps developers build LLM applications easily."
    ),
    Document(
        page_content="Chroma is a vector database optimized for LLM-based search."
    ),
    Document(
        page_content="Embeddings convert text into high-dimensional vectors."
    ),
    Document(
        page_content="Google provides powerful embedding models through Gemini."
    ),
]

# If using OpenAIEmbeddings
# Step 2: Initialize OpenAI Embedding Model
# embedding_model = OpenAIEmbeddings()

# If using GoogleGenerativeAIEmbeddings
# Step 2: Initialize Gemini Embedding Model
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004"
)

# Step 3: Create Chroma Vector Store
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

# Step 4: Convert Vector Store into a Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

query = "What is Chroma used for?"

# Retriever Search
results = retriever.invoke(query)

print("Retriever Results:")
for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)

# Similarity Search
results = vectorstore.similarity_search(query, k=2)

print("\n\nSimilarity Search Results:")
for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)

# Maximum Marginal Retriever (MMR)


In [ ]:
# If using OpenAIEmbeddings
# from langchain_openai import OpenAIEmbeddings

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from dotenv import load_dotenv

load_dotenv()

# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

# If using OpenAIEmbeddings
# Initialize embedding model
# embedding_model = OpenAIEmbeddings()

# If using GoogleGenerativeAIEmbeddings
# Initialize Gemini Embeddings
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004"
)

# Create FAISS Vector Store
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

# Enable MMR Retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "lambda_mult": 0.5,
    },
)

query = "What is LangChain?"

results = retriever.invoke(query)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)

# Multi-Query Retriever


In [3]:
# If using OpenAIEmbeddings and ChatOpenAI
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI,
)

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain.retrievers.multi_query import MultiQueryRetriever
from dotenv import load_dotenv

load_dotenv()

# Relevant health & wellness documents
all_docs = [
    Document(
        page_content="Regular walking boosts heart health and can reduce symptoms of depression.",
        metadata={"source": "H1"},
    ),
    Document(
        page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.",
        metadata={"source": "H2"},
    ),
    Document(
        page_content="Deep sleep is crucial for cellular repair and emotional regulation.",
        metadata={"source": "H3"},
    ),
    Document(
        page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.",
        metadata={"source": "H4"},
    ),
    Document(
        page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.",
        metadata={"source": "H5"},
    ),
    Document(
        page_content="The solar energy system in modern homes helps balance electricity demand.",
        metadata={"source": "I1"},
    ),
    Document(
        page_content="Python balances readability with power, making it a popular system design language.",
        metadata={"source": "I2"},
    ),
    Document(
        page_content="Photosynthesis enables plants to produce energy by converting sunlight.",
        metadata={"source": "I3"},
    ),
    Document(
        page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.",
        metadata={"source": "I4"},
    ),
    Document(
        page_content="Black holes bend spacetime and store immense gravitational energy.",
        metadata={"source": "I5"},
    ),
]

# If using OpenAIEmbeddings
# Initialize OpenAI Embedding Model
# embedding_model = OpenAIEmbeddings()

# If using GoogleGenerativeAIEmbeddings
# Initialize Gemini Embedding Model
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004"
)

# Create FAISS Vector Store
vectorstore = FAISS.from_documents(
    documents=all_docs,
    embedding=embedding_model
)

# Similarity Retriever
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# If using ChatOpenAI
# Initialize OpenAI LLM
# llm = ChatOpenAI(model="gpt-4o-mini")

# If using ChatGoogleGenerativeAI
# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Multi Query Retriever
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=llm,
)

# Query
query = "How to improve energy levels and maintain balance?"

# Retrieve Results
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

print("SIMILARITY RETRIEVER RESULTS")
print("=" * 80)

for i, doc in enumerate(similarity_results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)

print("\n" + "*" * 150)

print("\nMULTI QUERY RETRIEVER RESULTS")
print("=" * 80)

for i, doc in enumerate(multiquery_results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)

# Contextual Compression Retriever


In [4]:
# If using OpenAIEmbeddings and ChatOpenAI
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI,
)

from langchain_community.vectorstores import FAISS
from langchain.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain.retrievers.document_compressors import (
    LLMChainExtractor,
)
from langchain_core.documents import Document
from dotenv import load_dotenv

load_dotenv()

# Documents
docs = [
    Document(
        page_content="""
        The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year.
        The rocks date back millions of years.
        """,
        metadata={"source": "Doc1"},
    ),
    Document(
        page_content="""
        In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal.
        Siege weapons were often used to breach castle walls.
        """,
        metadata={"source": "Doc2"},
    ),
    Document(
        page_content="""
        Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets.
        NBA is now a global league.
        """,
        metadata={"source": "Doc3"},
    ),
    Document(
        page_content="""
        The history of cinema began in the late 1800s.
        Silent films were the earliest form.
        Thomas Edison was among the pioneers.
        Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design.
        """,
        metadata={"source": "Doc4"},
    ),
]

# If using OpenAIEmbeddings
# Initialize OpenAI Embedding Model
# embedding_model = OpenAIEmbeddings()

# If using GoogleGenerativeAIEmbeddings
# Initialize Gemini Embedding Model
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004"
)

# FAISS Vector Store
vectorstore = FAISS.from_documents(
    docs,
    embedding_model
)

# Base Retriever
base_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

# If using ChatOpenAI
# Initialize OpenAI LLM
# llm = ChatOpenAI(model="gpt-4o-mini")

# If using ChatGoogleGenerativeAI
# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

# LLM Compressor
compressor = LLMChainExtractor.from_llm(llm)

# Contextual Compression Retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor,
)

# Query
query = "What is photosynthesis?"

compressed_results = compression_retriever.invoke(query)

for i, doc in enumerate(compressed_results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)